# Búsqueda de la mejor combinación de variables para la Identidad organizacional

Prueba combinaciones de variables y número de clusters (KMeans) y califica cada una por:

| Criterio | Métrica | Qué significa |
|---|---|---|
| **Visibilidad** | silhouette 2D en los ejes de la gráfica | ¿se distinguen los clusters a simple vista? |
| **Explicabilidad** | exactitud de un árbol de decisión de profundidad 3 | ¿se puede describir cada identidad con 2–3 reglas simples? |
| **Parsimonia** | número de variables | entre menos, más fácil de interpretar |
| **Calidad** | silhouette en el espacio de las variables | ¿los clusters son compactos y separados? |
| **Estabilidad** | ARI entre corridas con submuestras | ¿salen los mismos clusters si cambian un poco los datos? |

La visibilidad se mide en dos planos:
- **Ejes actuales** de la gráfica 11 (log ingresos × log costo por beneficiario).
- **Mejores ejes**: el par de variables donde mejor se distinguen los clusters (sugerencia para cambiar la gráfica).

In [ ]:
import itertools
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text

warnings.filterwarnings("ignore")

# =========================================================
# CONFIGURACIÓN
# =========================================================
RUTA = "/Users/alanrobles/Documents/ALAN/GEOSTATS/dontarias/dataet"
LLAVE = ["Año", "Folio"]

RANGO_K = [3, 4, 5]               # número de identidades a probar (la definición original usa 4)
MIN_VARS, MAX_VARS = 2, 4         # tamaño de las combinaciones
PERMITIR_REDUNDANTES = False      # True = también prueba combinaciones con variables redundantes
MIN_TAMANO_CLUSTER = 0.05         # un cluster con menos del 5 % de las organizaciones se penaliza
N_MUESTRA = 3000                  # filas para silhouette / árbol (acelera sin cambiar el resultado)
SEMILLA = 42

# Peso de cada criterio en la calificación final (suman 1). Ajústalos según lo que más te importe.
PESOS = {
    "visibilidad":    0.30,
    "explicabilidad": 0.25,
    "parsimonia":     0.20,
    "calidad":        0.15,
    "estabilidad":    0.10,
}
# ¿La visibilidad se mide en los ejes actuales de la gráfica 11 o en los mejores ejes posibles?
#   "actuales" → buscas clusters que se vean bien SIN cambiar la gráfica
#   "mejores"  → aceptas cambiar los ejes de la gráfica si así se distinguen mejor
VISIBILIDAD_EN = "mejores"

## 1. Construcción de variables (misma lógica que el notebook original)

In [ ]:
t0 = time.time()
generales = pd.read_excel(f"{RUTA}/Generales.xlsx")
donativos = pd.read_excel(f"{RUTA}/Ingreso por donativos.xlsx")
relacionados = pd.read_excel(f"{RUTA}/Ingresos relacionados.xlsx")
no_relacionados = pd.read_excel(f"{RUTA}/Ingresos no relacionados.xlsx")
gastos = pd.read_excel(f"{RUTA}/Gastos.xlsx")
destino = pd.read_excel(f"{RUTA}/Destino de donativos.xlsx")

df = (
    donativos.groupby(LLAVE)[["Monto efectivo", "Monto especie"]].sum().reset_index()
    .merge(relacionados.groupby(LLAVE)["Monto"].sum().rename("Ingresos relacionados").reset_index(), on=LLAVE, how="outer")
    .merge(no_relacionados.groupby(LLAVE)["Monto"].sum().rename("Ingresos no relacionados").reset_index(), on=LLAVE, how="outer")
    .fillna(0)
)
df["Total Ingresos Anio Fiscal"] = df[["Monto efectivo", "Monto especie", "Ingresos relacionados", "Ingresos no relacionados"]].sum(axis=1)
df["Dependencia de donativos"] = ((df["Monto efectivo"] + df["Monto especie"]) / df["Total Ingresos Anio Fiscal"]).fillna(0)

gastos["Total Gastos"] = gastos[["Monto nacional operación", "Monto nacional admin",
                                 "Monto extranjero operación", "Monto extranjero admin"]].sum(axis=1)
df = df.merge(gastos.groupby(LLAVE)["Total Gastos"].sum().reset_index(), on=LLAVE, how="left")
df["Total Gastos"] = df["Total Gastos"].fillna(0)
df["Sustentabilidad"] = ((df["Total Ingresos Anio Fiscal"] - df["Total Gastos"]) / df["Total Ingresos Anio Fiscal"])
df["Sustentabilidad"] = df["Sustentabilidad"].replace([np.inf, -np.inf], np.nan).fillna(0).clip(-2, 1)

benef = destino.groupby(LLAVE)["Número de beneficiados"].sum().rename("Total Beneficiarios").reset_index()
df = df.merge(benef, on=LLAVE, how="left")
df["Total Beneficiarios"] = df["Total Beneficiarios"].fillna(0)
df["Costo por beneficiario"] = np.where(df["Total Beneficiarios"] > 0,
                                        df["Total Ingresos Anio Fiscal"] / df["Total Beneficiarios"], np.nan)

df = df[(df["Total Ingresos Anio Fiscal"] >= 100) & (df["Año"] != 2026)]

# Quintil por año (1 = más ingresos). Escrito así funciona en pandas 2 y 3.
df["Quintil de ingresos"] = (
    df.groupby("Año")["Total Ingresos Anio Fiscal"]
      .transform(lambda s: pd.qcut(s.rank(method="first", ascending=False), 5, labels=False) + 1)
      .astype(int)
)

# Rfc para poder unir con el dashboard
df = df.merge(generales[["Año", "Folio", "Rfc"]].drop_duplicates(["Año", "Folio"]), on=["Año", "Folio"], how="left")

df_cluster = df.dropna(subset=["Costo por beneficiario"]).reset_index(drop=True)
df_cluster = df_cluster[df_cluster["Costo por beneficiario"] > 0].reset_index(drop=True)
print(f"Organizaciones-año para clustering: {len(df_cluster):,}  ({time.time()-t0:.0f}s)")

## 2. Variables candidatas y grupos de redundancia

- `ingresos`, `beneficiarios` y `costo` cumplen **log costo = log ingresos − log beneficiarios**:
  usar las tres no agrega información (solo duplica peso). Con 2 de ellas basta.
- `quintil` es el ingreso discretizado: no se combina con `ingresos`.

In [ ]:
# nombre corto → (columna transformada para el modelo, columna original para las reglas, etiqueta)
VARS = {
    "ingresos":        ("log_ingresos", "Total Ingresos Anio Fiscal", "Total de ingresos (log)"),
    "beneficiarios":   ("log_benef",    "Total Beneficiarios",        "Total beneficiarios (log)"),
    "costo":           ("log_costo",    "Costo por beneficiario",     "Costo por beneficiario (log)"),
    "dependencia":     ("dependencia",  "Dependencia de donativos",   "Dependencia de donativos"),
    "sustentabilidad": ("sustent",      "Sustentabilidad",            "Sustentabilidad"),
    "quintil":         ("quintil",      "Quintil de ingresos",        "Quintil de ingresos"),
}
df_cluster["log_ingresos"] = np.log10(df_cluster["Total Ingresos Anio Fiscal"])
df_cluster["log_benef"]    = np.log10(df_cluster["Total Beneficiarios"])
df_cluster["log_costo"]    = np.log10(df_cluster["Costo por beneficiario"])
df_cluster["dependencia"]  = df_cluster["Dependencia de donativos"]
df_cluster["sustent"]      = df_cluster["Sustentabilidad"]
df_cluster["quintil"]      = df_cluster["Quintil de ingresos"]

def es_redundante(combo):
    s = set(combo)
    return {"ingresos", "beneficiarios", "costo"} <= s or {"quintil", "ingresos"} <= s

# Ejes posibles para la gráfica (variables continuas; el quintil no sirve como eje)
EJES_CANDIDATOS = [v for v in VARS if v != "quintil"]
EJES_ACTUALES = ("ingresos", "costo")          # gráfica 11 de hoy

combos = [c for n in range(MIN_VARS, MAX_VARS + 1) for c in itertools.combinations(VARS, n)
          if PERMITIR_REDUNDANTES or not es_redundante(c)]
ORIGINAL = ("quintil", "ingresos", "beneficiarios", "costo", "dependencia", "sustentabilidad")
combos.append(ORIGINAL)   # referencia: la combinación que usas hoy
print(f"Combinaciones a probar: {len(combos)} × {len(RANGO_K)} valores de k = {len(combos)*len(RANGO_K)} modelos")

## 3. Métricas

In [ ]:
rng = np.random.default_rng(SEMILLA)
idx_m = rng.choice(len(df_cluster), size=min(N_MUESTRA, len(df_cluster)), replace=False)

# Coordenadas estandarizadas de cada posible eje (para medir qué tanto se ve en una gráfica 2D)
Z_EJES = {v: StandardScaler().fit_transform(df_cluster[[VARS[v][0]]]).ravel() for v in EJES_CANDIDATOS}

def silueta_2d(labels, ejes):
    XY = np.column_stack([Z_EJES[ejes[0]][idx_m], Z_EJES[ejes[1]][idx_m]])
    lab = labels[idx_m]
    return silhouette_score(XY, lab) if len(np.unique(lab)) > 1 else -1

def explicabilidad(labels, combo):
    """¿Qué tan bien reproduce los clusters un árbol de profundidad 3 con las variables ORIGINALES?
    Alta = cada identidad se describe con pocas reglas del tipo 'ingresos > X y dependencia < Y'."""
    X = df_cluster.loc[idx_m, [VARS[v][1] for v in combo]].values
    arbol = DecisionTreeClassifier(max_depth=3, random_state=SEMILLA)
    return cross_val_score(arbol, X, labels[idx_m], cv=3).mean()

def estabilidad(X, k, labels_full):
    """ARI promedio entre el modelo completo y modelos ajustados con submuestras del 70 %."""
    aris = []
    for s in range(2):
        r = np.random.default_rng(SEMILLA + 1 + s)
        sub = r.choice(len(X), size=int(0.7 * len(X)), replace=False)
        km = KMeans(n_clusters=k, random_state=SEMILLA + 1 + s, n_init=4).fit(X[sub])
        aris.append(adjusted_rand_score(labels_full, km.predict(X)))
    return float(np.mean(aris))

## 4. Búsqueda

In [ ]:
t0 = time.time()
filas, MODELOS = [], {}
for combo in combos:
    cols = [VARS[v][0] for v in combo]
    X = StandardScaler().fit_transform(df_cluster[cols].values)
    for k in RANGO_K:
        km = KMeans(n_clusters=k, random_state=SEMILLA, n_init=10).fit(X)
        lab = km.labels_
        tamanos = np.bincount(lab) / len(lab)

        vis_actual = silueta_2d(lab, EJES_ACTUALES)
        planos = {par: silueta_2d(lab, par) for par in itertools.combinations(EJES_CANDIDATOS, 2)}
        mejor_par = max(planos, key=planos.get)

        fila = {
            "variables": " + ".join(combo),
            "n_vars": len(combo),
            "k": k,
            "redundante": es_redundante(combo),
            "vis_ejes_actuales": vis_actual,
            "vis_mejores_ejes": planos[mejor_par],
            "mejores_ejes": f"{mejor_par[0]} × {mejor_par[1]}",
            "explicabilidad": explicabilidad(lab, combo),
            "calidad": silhouette_score(X[idx_m], lab[idx_m]),
            "estabilidad": estabilidad(X, k, lab),
            "cluster_min_%": tamanos.min() * 100,
        }
        filas.append(fila)
        MODELOS[(combo, k)] = (lab, mejor_par)
print(f"Listo en {time.time()-t0:.0f}s")

res = pd.DataFrame(filas)

## 5. Calificación final
Cada métrica se lleva a 0–1 (min-max entre todos los modelos) y se combina con `PESOS`.

In [ ]:
def norm(s):
    return (s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s * 0 + 1

col_vis = "vis_mejores_ejes" if VISIBILIDAD_EN == "mejores" else "vis_ejes_actuales"
res["parsimonia"] = 1 - (res["n_vars"] - MIN_VARS) / (len(ORIGINAL) - MIN_VARS)
res["score"] = (
    PESOS["visibilidad"]    * norm(res[col_vis])
    + PESOS["explicabilidad"] * norm(res["explicabilidad"])
    + PESOS["parsimonia"]     * res["parsimonia"]
    + PESOS["calidad"]        * norm(res["calidad"])
    + PESOS["estabilidad"]    * norm(res["estabilidad"])
)
res.loc[res["cluster_min_%"] < MIN_TAMANO_CLUSTER * 100, "score"] *= 0.5   # cluster casi vacío
res = res.sort_values("score", ascending=False).reset_index(drop=True)

pd.set_option("display.width", 220, "display.max_columns", 30)
cols_ver = ["variables", "k", "score", col_vis, "mejores_ejes", "vis_ejes_actuales",
            "explicabilidad", "calidad", "estabilidad", "cluster_min_%"]
print("TOP 15")
print(res[cols_ver].head(15).round(3).to_string())
print("\nCOMBINACIÓN ORIGINAL (referencia)")
print(res[res["variables"] == " + ".join(ORIGINAL)][cols_ver].round(3).to_string())

res.to_excel("busqueda_clusters_resultados.xlsx", index=False)

## 6. Comparación visual: original vs. los mejores
Cada columna es un modelo; arriba en los ejes actuales de la gráfica 11, abajo en sus mejores ejes.

In [ ]:
COLORES = ["#8B2C1A", "#003153", "#B15E2E", "#2C2C2C", "#8FA9B8", "#C9A227"]

def graficar(ax, lab, ejes, titulo):
    sub = idx_m
    for c in np.unique(lab):
        m = lab[sub] == c
        ax.scatter(df_cluster[VARS[ejes[0]][0]].values[sub][m], df_cluster[VARS[ejes[1]][0]].values[sub][m],
                   s=6, alpha=0.45, color=COLORES[c % len(COLORES)], label=f"C{c}", linewidths=0)
    ax.set_xlabel(VARS[ejes[0]][2], fontsize=8); ax.set_ylabel(VARS[ejes[1]][2], fontsize=8)
    ax.set_title(titulo, fontsize=8); ax.tick_params(labelsize=7)

orig_k4 = res[(res["variables"] == " + ".join(ORIGINAL)) & (res["k"] == 4)].iloc[0]
mostrar = [orig_k4] + [r for _, r in res.head(4).iterrows()]
fig, axes = plt.subplots(2, len(mostrar), figsize=(4 * len(mostrar), 7.5))
for j, r in enumerate(mostrar):
    combo = tuple(r["variables"].split(" + "))
    lab, mejor_par = MODELOS[(combo, r["k"])]
    tag = "ORIGINAL" if j == 0 else f"#{j}"
    graficar(axes[0, j], lab, EJES_ACTUALES, f"{tag} · k={r['k']}\n{r['variables']}\nvis actual={r['vis_ejes_actuales']:.2f}")
    graficar(axes[1, j], lab, mejor_par, f"mejores ejes · vis={r['vis_mejores_ejes']:.2f}")
plt.tight_layout()
plt.savefig("busqueda_clusters_comparacion.png", dpi=130)
plt.show()

## 7. Perfil y reglas de la combinación elegida
Cambia `ELEGIDA` por el renglón del ranking que quieras revisar (0 = el mejor).

In [ ]:
ELEGIDA = 0
r = res.iloc[ELEGIDA]
combo, k = tuple(r["variables"].split(" + ")), int(r["k"])
lab, mejor_par = MODELOS[(combo, k)]
df_cluster["Cluster"] = lab

print(f"Combinación: {r['variables']}  ·  k={k}  ·  mejores ejes: {r['mejores_ejes']}\n")
perfil = df_cluster.groupby("Cluster").agg(
    n=("Cluster", "size"),
    ingresos_mediana=("Total Ingresos Anio Fiscal", "median"),
    beneficiarios_mediana=("Total Beneficiarios", "median"),
    costo_benef_mediana=("Costo por beneficiario", "median"),
    dependencia_media=("Dependencia de donativos", "mean"),
    sustentabilidad_media=("Sustentabilidad", "mean"),
    quintil_moda=("Quintil de ingresos", lambda s: s.mode().iloc[0]),
)
perfil["%"] = (perfil["n"] / perfil["n"].sum() * 100).round(1)
print(perfil.round(2).to_string())

# Reglas: el árbol resume cada cluster con cortes en las unidades originales
arbol = DecisionTreeClassifier(max_depth=3, random_state=SEMILLA).fit(
    df_cluster[[VARS[v][1] for v in combo]], lab)
print(f"\nReglas (árbol de profundidad 3, exactitud {arbol.score(df_cluster[[VARS[v][1] for v in combo]], lab):.1%}):")
print(export_text(arbol, feature_names=[VARS[v][1] for v in combo], decimals=2,
                  class_names=[f"C{c}" for c in range(k)]))

## 8. Guardar para el dashboard
Pon el nombre de cada identidad según el perfil de arriba y cambia `GUARDAR = True`.
Genera el mismo `Identidad organizacional.xlsx` que usa `dashboard_donatarias.py`.

In [ ]:
GUARDAR = False
ETIQUETAS_CLUSTER = {
    0: "Salamandra",
    1: "Axolote",
    2: "Axolote con rendimiento de Salamandra",
    3: "Crisálida",
}

if GUARDAR:
    df_cluster["Identidad organizacional"] = df_cluster["Cluster"].map(ETIQUETAS_CLUSTER)
    salida = df_cluster[["Año", "Folio", "Rfc", "Identidad organizacional", "Cluster",
                         "Quintil de ingresos", "Total Ingresos Anio Fiscal", "Total Beneficiarios",
                         "Costo por beneficiario", "Dependencia de donativos", "Sustentabilidad"]]
    salida.to_excel(f"{RUTA}/Identidad organizacional.xlsx", index=False)
    print(f"Guardado ({len(salida):,} filas) · ejes sugeridos para la gráfica 11: {r['mejores_ejes']}")